# MathCursor — Fine-tuning XLM-RoBERTa pour détection NER de zones math

**Tout dans un dossier Drive unique.**

Avant de lancer :
1. Crée un dossier dans ton Google Drive, ex : `MyDrive/mathcursor`
2. Dépose-y `train.jsonl`, `val.jsonl`, `test.jsonl`
3. Runtime → Modifier le type d'exécution → **GPU T4**
4. Runtime → Tout exécuter

Tous les modèles, logs, métriques et archives seront écrits dans ce même dossier.

## 1. Mount Drive + vérification du dossier

Change `DRIVE_FOLDER` si tu utilises un autre nom.

In [ ]:
from google.colab import drive
import os

DRIVE_FOLDER = 'mathcursor'

drive.mount('/content/drive')
WORKDIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'

if not os.path.isdir(WORKDIR):
    msg = (
        f'Dossier introuvable : {WORKDIR}. '
        f'Crée-le dans MyDrive et dépose-y train.jsonl, val.jsonl, test.jsonl'
    )
    raise RuntimeError(msg)

# Splits obligatoires
for name in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    path = os.path.join(WORKDIR, name)
    if not os.path.isfile(path):
        raise RuntimeError(f'Fichier manquant : {path}')
    size = os.path.getsize(path) / 1024
    print(f'OK {name}: {size:.1f} Ko')

# Extensions optionnelles — concaténées au train si présentes
EXTENSIONS = [
    'extension_v3_fixtures.jsonl',
    'extension_v3_superscript2.jsonl',
    'extension_v3_false_positives.jsonl',
    'extension_v4_keywords.jsonl',
    'extension_v5_quant_letters.jsonl',
    'extension_v6_recent_features.jsonl',
    'extension_v6_1_targeted.jsonl',
]
for name in EXTENSIONS:
    path = os.path.join(WORKDIR, name)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1024
        print(f'OK {name}: {size:.1f} Ko (extension)')
    else:
        print(f'-- {name}: absent (skip)')

print()
print(f'WORKDIR = {WORKDIR}')

## 2. Installation des dépendances

In [ ]:
# Colab pré-installe diffusers qui casse l'environnement, on le dégage
!pip uninstall -y diffusers

# Dernières versions de tout
!pip install -q -U transformers datasets accelerate seqeval \
    "optimum[onnxruntime]" onnx onnxruntime sentencepiece protobuf

import importlib
for pkg in ['transformers', 'datasets', 'seqeval', 'accelerate', 'onnx', 'onnxruntime', 'optimum']:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'OK {pkg} {v}')
    except ImportError:
        print(f'MANQUANT {pkg}')

print('\n⚠️  Runtime → Restart session → puis Run all')

Found existing installation: diffusers 0.37.1
Uninstalling diffusers-0.37.1:
  Successfully uninstalled diffusers-0.37.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 148.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 25.1 MB/s eta 0:00:00
OK transformers 4.57.6
OK datasets 4.8.4
OK seqeval ?
OK accelerate 1.13.0
OK 

## 3. Paramètres

In [ ]:
MODEL_NAME = 'xlm-roberta-base'

OUTPUT_DIR = os.path.join(WORKDIR, 'model-pt')
ONNX_DIR = os.path.join(WORKDIR, 'model-onnx')
QUANT_DIR = os.path.join(WORKDIR, 'model-onnx-int8')
LOG_DIR = os.path.join(WORKDIR, 'logs')

MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
NUM_EPOCHS = 4
WEIGHT_DECAY = 0.01

LABELS = ['O', 'B-MATH', 'I-MATH']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

## 4. Chargement et conversion spans → BIO

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def spans_to_char_labels(text, spans):
    labels = ['O'] * len(text)
    for span in spans:
        for i in range(span['start'], span['end']):
            if i < len(labels):
                labels[i] = 'MATH'
    return labels

def to_dataset_dict(examples):
    return {
        'text': [e['text'] for e in examples],
        'spans': [e['spans'] for e in examples],
        'lang': [e['lang'] for e in examples],
    }

train_data = load_jsonl(os.path.join(WORKDIR, 'train.jsonl'))
val_data = load_jsonl(os.path.join(WORKDIR, 'val.jsonl'))
test_data = load_jsonl(os.path.join(WORKDIR, 'test.jsonl'))

# Concat des extensions présentes au train (val et test inchangés)
for name in EXTENSIONS:
    path = os.path.join(WORKDIR, name)
    if os.path.isfile(path):
        ext = load_jsonl(path)
        train_data += ext
        print(f'+ {name}: {len(ext)} lignes ajoutées au train')

print()
print(f'Train total: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

datasets = DatasetDict({
    'train': Dataset.from_dict(to_dataset_dict(train_data)),
    'validation': Dataset.from_dict(to_dataset_dict(val_data)),
    'test': Dataset.from_dict(to_dataset_dict(test_data)),
})

print(datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'spans', 'lang'],
        num_rows: 6399
    })
    validation: Dataset({
        features: ['text', 'spans', 'lang'],
        num_rows: 798
    })
    test: Dataset({
        features: ['text', 'spans', 'lang'],
        num_rows: 803
    })
})


## 5. Tokenisation et alignement BIO

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=False)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
        padding=False,
    )

    all_labels = []
    for i, text in enumerate(examples['text']):
        char_labels = spans_to_char_labels(text, examples['spans'][i])
        offsets = tokenized['offset_mapping'][i]

        token_labels = []
        prev_label = 'O'
        for (start, end) in offsets:
            if start == end:
                token_labels.append(-100)
                continue
            char_label = char_labels[start] if start < len(char_labels) else 'O'
            if char_label == 'MATH':
                if prev_label != 'MATH':
                    token_labels.append(LABEL2ID['B-MATH'])
                else:
                    token_labels.append(LABEL2ID['I-MATH'])
                prev_label = 'MATH'
            else:
                token_labels.append(LABEL2ID['O'])
                prev_label = 'O'
        all_labels.append(token_labels)

    tokenized['labels'] = all_labels
    tokenized.pop('offset_mapping')
    return tokenized

tokenized_datasets = datasets.map(
    tokenize_and_align,
    batched=True,
    remove_columns=['text', 'spans', 'lang'],
)

print('Tokenisation OK')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/6399 [00:00<?, ? examples/s]

Map:   0%|          | 0/798 [00:00<?, ? examples/s]

Map:   0%|          | 0/803 [00:00<?, ? examples/s]

Tokenisation OK


## 6. Modèle, métriques, Trainer

In [ ]:
import numpy as np
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [ID2LABEL[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=LOG_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_10171/1848867961.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 7. Entraînement

~10-15 min sur GPU T4 pour 6000 lignes × 4 époques. Checkpoints écrits directement dans Drive.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.009400,0.002179,0.991678,0.990305,0.990991
2,0.004600,0.000212,1.000000,1.000000,1.000000
3,0.000600,0.000280,1.000000,1.000000,1.000000
4,0.000400,0.000016,1.000000,1.000000,1.000000


TrainOutput(global_step=1600, training_loss=0.019029055981663986, metrics={'train_runtime': 148.4833, 'train_samples_per_second': 172.383, 'train_steps_per_second': 10.776, 'total_flos': 512716334217354.0, 'train_loss': 0.019029055981663986, 'epoch': 4.0})

## 8. Évaluation sur test set

In [ ]:
predictions, labels, metrics = trainer.predict(tokenized_datasets['test'])
predictions = np.argmax(predictions, axis=2)

true_labels = [
    [ID2LABEL[l] for l in label if l != -100]
    for label in labels
]
true_predictions = [
    [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

print('=== MÉTRIQUES TEST ===')
print(f'Precision: {precision_score(true_labels, true_predictions):.4f}')
print(f'Recall:    {recall_score(true_labels, true_predictions):.4f}')
print(f'F1:        {f1_score(true_labels, true_predictions):.4f}')
print()
print(classification_report(true_labels, true_predictions))

metrics_out = {
    'precision': float(precision_score(true_labels, true_predictions)),
    'recall': float(recall_score(true_labels, true_predictions)),
    'f1': float(f1_score(true_labels, true_predictions)),
}
with open(os.path.join(WORKDIR, 'metrics.json'), 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'\nMétriques écrites dans {WORKDIR}/metrics.json')

=== MÉTRIQUES TEST ===
Precision: 0.9949
Recall:    0.9962
F1:        0.9956

              precision    recall  f1-score   support

        MATH       0.99      1.00      1.00       788

   micro avg       0.99      1.00      1.00       788
   macro avg       0.99      1.00      1.00       788
weighted avg       0.99      1.00      1.00       788


Métriques écrites dans /content/drive/MyDrive/mathcursor/metrics.json


## 9. Test inférence sur exemples concrets

In [ ]:
from transformers import pipeline

ner = pipeline(
    'token-classification',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy='simple',
    device=0,
)

test_sentences = [
    "on a limite f(x) qd x -> +inf",
    "on a a + b = 2",
    "the sequence un converges to 0",
    "il a une grande puissance de travail",
    "soit f(x) = 2x+1 et g(x) = 3x-1, alors f(g(x)) = 6x+1",
    "the matrix A is invertible with det(A) = 5",
    "on calcule V(x+1) pour x dans R",
    "j'ai oublie mon cahier d'exercices",
]

for sentence in test_sentences:
    results = ner(sentence)
    print(f'\n{sentence!r}')
    if not results:
        print('  (aucune zone math détectée)')
    for r in results:
        span_text = sentence[r['start']:r['end']]
        print(f"  [{r['start']:3d}-{r['end']:3d}] conf={r['score']:.2f} : {span_text!r}")

Device set to use cuda:0



'on a limite f(x) qd x -> +inf'
  [ 12- 29] conf=1.00 : 'f(x) qd x -> +inf'

'on a a + b = 2'
  [  5- 14] conf=1.00 : 'a + b = 2'

'the sequence un converges to 0'
  [ 13- 15] conf=1.00 : 'un'
  [ 26- 30] conf=0.80 : 'to 0'

'il a une grande puissance de travail'
  (aucune zone math détectée)

'soit f(x) = 2x+1 et g(x) = 3x-1, alors f(g(x)) = 6x+1'
  [  5- 16] conf=1.00 : 'f(x) = 2x+1'
  [ 20- 31] conf=1.00 : 'g(x) = 3x-1'
  [ 39- 53] conf=1.00 : 'f(g(x)) = 6x+1'

'the matrix A is invertible with det(A) = 5'
  [ 11- 12] conf=1.00 : 'A'
  [ 32- 42] conf=1.00 : 'det(A) = 5'

'on calcule V(x+1) pour x dans R'
  [ 11- 31] conf=1.00 : 'V(x+1) pour x dans R'

"j'ai oublie mon cahier d'exercices"
  (aucune zone math détectée)


In [ ]:
# Phrases écrites par un humain, hors distribution du dataset
real_world_sentences = [
    # Phrases de prof / élève, pas dans le dataset
    "Pour resoudre cette equation, on factorise : x^2 - 5x + 6 = (x-2)(x-3)",
    "La derivee de sin(3x) vaut 3cos(3x), donc integrale = -cos(3x)/3",
    "Si la suite (un) est telle que un+1 - un = 2, alors elle est arithmetique",
    "Le produit scalaire de deux vecteurs orthogonaux est nul",
    "J'ai pas compris pourquoi cos pi = -1 et pas 1",
    "Avec h tendant vers 0, f(x+h)-f(x) divise par h donne f'(x)",
    # Cas piégeux
    "Pour tout epsilon positif il existe n a partir duquel |un - L| < epsilon",
    "Mon frere a sin aine comme prenom",
    "La racine carree d'un nombre negatif n'existe pas dans R",
    "Faut trouver x tel que ln x egale 2",
    # Style SMS / raccourcis d'élève
    "soit f : x -> 2x+1, calc f(3)",
    "lim en + inf de 1/n^2 c 0",
    "dérivée de e^x c toujours e^x frr",
]

print("=== TEST HORS DISTRIBUTION ===\n")
for sentence in real_world_sentences:
    results = ner(sentence)
    print(f'{sentence!r}')
    if not results:
        print('  (rien détecté)')
    for r in results:
        span_text = sentence[r['start']:r['end']]
        print(f"  [{r['start']:3d}-{r['end']:3d}] conf={r['score']:.2f} : {span_text!r}")
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== TEST HORS DISTRIBUTION ===

'Pour resoudre cette equation, on factorise : x^2 - 5x + 6 = (x-2)(x-3)'
  [ 45- 70] conf=1.00 : 'x^2 - 5x + 6 = (x-2)(x-3)'

'La derivee de sin(3x) vaut 3cos(3x), donc integrale = -cos(3x)/3'
  [ 14- 21] conf=1.00 : 'sin(3x)'
  [ 27- 36] conf=1.00 : '3cos(3x),'
  [ 42- 64] conf=1.00 : 'integrale = -cos(3x)/3'

'Si la suite (un) est telle que un+1 - un = 2, alors elle est arithmetique'
  [ 12- 16] conf=1.00 : '(un)'
  [ 31- 44] conf=1.00 : 'un+1 - un = 2'
  [ 61- 73] conf=1.00 : 'arithmetique'

'Le produit scalaire de deux vecteurs orthogonaux est nul'
  (rien détecté)

"J'ai pas compris pourquoi cos pi = -1 et pas 1"
  [ 26- 37] conf=1.00 : 'cos pi = -1'
  [ 41- 46] conf=0.84 : 'pas 1'

"Avec h tendant vers 0, f(x+h)-f(x) divise par h donne f'(x)"
  [  5- 21] conf=1.00 : 'h tendant vers 0'
  [ 23- 41] conf=1.00 : 'f(x+h)-f(x) divise'
  [ 54- 59] conf=1.00 : "f'(x)"

'Pour tout epsilon positif il existe n a partir duquel |un - L| < epsilon'
  [ 26- 37] c

## 10. Export ONNX + quantization int8 (dans Drive)

In [ ]:
from optimum.onnxruntime import ORTModelForTokenClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

ort_model = ORTModelForTokenClassification.from_pretrained(OUTPUT_DIR, export=True)
ort_model.save_pretrained(ONNX_DIR)
tokenizer.save_pretrained(ONNX_DIR)

print('ONNX exporté dans', ONNX_DIR)
for f in sorted(os.listdir(ONNX_DIR)):
    size = os.path.getsize(os.path.join(ONNX_DIR, f)) / (1024 * 1024)
    print(f'  {f}: {size:.1f} Mo')

quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)
tokenizer.save_pretrained(QUANT_DIR)

print('\nQuantizé int8 dans', QUANT_DIR)
for f in sorted(os.listdir(QUANT_DIR)):
    size = os.path.getsize(os.path.join(QUANT_DIR, f)) / (1024 * 1024)
    print(f'  {f}: {size:.1f} Mo')

`torch_dtype` is deprecated! Use `dtype` instead!
The tokenizer you are loading from '/content/drive/MyDrive/mathcursor/model-pt' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/mathcursor/model-pt' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/mathcursor/model-pt' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This 

ONNX exporté dans /content/drive/MyDrive/mathcursor/model-onnx
  config.json: 0.0 Mo
  model.onnx: 1058.6 Mo
  sentencepiece.bpe.model: 4.8 Mo
  special_tokens_map.json: 0.0 Mo
  tokenizer.json: 16.3 Mo
  tokenizer_config.json: 0.0 Mo


The tokenizer you are loading from '/content/drive/MyDrive/mathcursor/model-onnx' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/mathcursor/model-onnx' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



Quantizé int8 dans /content/drive/MyDrive/mathcursor/model-onnx-int8
  config.json: 0.0 Mo
  model_quantized.onnx: 265.4 Mo
  ort_config.json: 0.0 Mo
  sentencepiece.bpe.model: 4.8 Mo
  special_tokens_map.json: 0.0 Mo
  tokenizer.json: 16.3 Mo
  tokenizer_config.json: 0.0 Mo


## 11. Validation du modèle quantizé

In [ ]:
import time

ort_quant = ORTModelForTokenClassification.from_pretrained(
    QUANT_DIR,
    file_name='model_quantized.onnx',
    provider='CPUExecutionProvider',
)

ner_quant = pipeline(
    'token-classification',
    model=ort_quant,
    tokenizer=tokenizer,
    aggregation_strategy='simple',
    device=-1,  # force CPU
)

for sentence in test_sentences[:4]:
    t0 = time.time()
    results = ner_quant(sentence)
    dt = (time.time() - t0) * 1000
    print(f'\n[{dt:.1f}ms] {sentence!r}')
    for r in results:
        span_text = sentence[r['start']:r['end']]
        print(f"  [{r['start']:3d}-{r['end']:3d}] conf={r['score']:.2f} : {span_text!r}")

Device set to use cpu



[13.9ms] 'on a limite f(x) qd x -> +inf'
  [ 12- 29] conf=1.00 : 'f(x) qd x -> +inf'

[9.2ms] 'on a a + b = 2'
  [  5- 14] conf=1.00 : 'a + b = 2'

[9.9ms] 'the sequence un converges to 0'
  [ 13- 15] conf=1.00 : 'un'
  [ 29- 30] conf=0.70 : '0'

[8.6ms] 'il a une grande puissance de travail'


## 12. Archive ZIP finale dans Drive

In [ ]:
import shutil

archive_base = os.path.join(WORKDIR, 'mathcursor-ner-v4')
shutil.make_archive(archive_base, 'zip', QUANT_DIR)
archive_path = archive_base + '.zip'
archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f'Archive créée : {archive_path} ({archive_size:.1f} Mo)')

print()
print('=== CONTENU FINAL DU DOSSIER DRIVE ===')
for f in sorted(os.listdir(WORKDIR)):
    path = os.path.join(WORKDIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f'  [file] {f}: {size:.2f} Mo')
    else:
        print(f'  [dir]  {f}/')

Archive créée : /content/drive/MyDrive/mathcursor/mathcursor-ner-int8.zip (208.5 Mo)

=== CONTENU FINAL DU DOSSIER DRIVE ===
  [file] mathcursor-ner-int8.zip: 208.54 Mo
  [file] metrics.json: 0.00 Mo
  [dir]  model-onnx/
  [dir]  model-onnx-int8/
  [dir]  model-pt/
  [file] test.jsonl: 0.10 Mo
  [file] train.jsonl: 0.78 Mo
  [file] val.jsonl: 0.10 Mo
